# Dota 2 Drafter - Interactive Machine Learning & Drafting Exploration

This notebook provides an interactive environment to explore the ML pipeline components:
1. **Data Ingestion & Cleaning**: Validating 5v5 draft records with `ml_pipeline.prepare_data`
2. **Leakage-Free Training**: Empirical-Bayes shrinkage and LightGBM model training with `ml_pipeline.train`
3. **Rigorous Evaluation**: Temporal split benchmarking against baseline models with `ml_pipeline.evaluate`
4. **Interactive Recommendations**: Real-time counter-pick scoring and synergy reasoning via `backend.app.scoring`

In [ ]:
import sys
from pathlib import Path

# Add project root and backend to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
backend_dir = PROJECT_ROOT / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

import pandas as pd
import numpy as np
print("Project root configured:", PROJECT_ROOT)

## Step 1: Clean and Prepare Match Records
Prepare raw match records into validated 5v5 match rows using `prepare_dataset`.

In [ ]:
from ml_pipeline.prepare_data import prepare_dataset

fixture_path = str(PROJECT_ROOT / "tests" / "fixtures" / "matches_fixture.csv")
out_clean_path = str(PROJECT_ROOT / "data" / "clean_matches_sample.csv")

clean_df = prepare_dataset(input_csv=fixture_path, output_csv=out_clean_path)
print(f"Cleaned {len(clean_df)} matches successfully.")
clean_df.head(3)

## Step 2: Time-Based Model Training & Learned Stacking Combiner
Train the LightGBM classifier with Empirical-Bayes shrinkage and fit combination weights on the validation slice.

In [ ]:
from ml_pipeline.train import train_pipeline

weights, metadata = train_pipeline(
    input_csv=fixture_path,
    output_dir=str(PROJECT_ROOT / "backend" / "model"),
    config_path=str(PROJECT_ROOT / "ml_pipeline" / "training_config.json"),
    version="v2.2.0"
)
print("Learned Combiner Weights:", weights)
print("Model Version:", metadata.get("version"))
print("Data Freshness:", metadata.get("data_freshness"))

## Step 3: Interactive Draft Counter-Pick Recommendation
Demonstrate role-aware counter-pick recommendations with confidence scoring and sample-size tracking.

In [ ]:
from app.model_loader import store
from app.scoring import get_counter_picks

# Ensure models are loaded in memory
store.load_models()

# Simulate a realistic draft scenario
recommendation = get_counter_picks(
    enemy_input=["Pudge", "Viper", "Sniper"],
    team_input=["Anti-Mage", "Puck"],
    bans_input=["Lion", "Shadow Fiend"],
    roles_needed=None,
    role_aware=True
)

print(f"Inferred Missing Roles: {recommendation.get('roles_needed')}")
print("\nTop 5 Counter-Pick Suggestions:")
for pick in recommendation["picks"][:5]:
    print(f"\n* [{pick['confidence'].upper():<6}] #{pick['id']:<3} {pick['name']:<18} | Score: {pick['score']:.4f}")
    print(f"  Roles: {', '.join(pick['roles'])}")
    print(f"  Sample Size: {pick['sample_size']} games")
    for r in pick['reasons'][:2]:
        print(f"  - {r}")